# Chapter 4: reproducible copy-trading risk analysis

This notebook reproduces the three quantitative findings reported in Chapter 4
(Figures 7–9) from the fixed OKX copy-trading snapshot dated **2 July 2026**.

**Publication boundary.** The input contains 200 researcher-assigned pseudonyms
(for example, `#017 Aster`) and excludes platform account IDs. The public `pnlRatio`
histories are treated as a return-index proxy rather than audited equity. The
analysis is descriptive: it cannot identify liquidation, deposits, withdrawals,
follower-level losses, prediction, or causation.

Run the cells from top to bottom. The final cell saves the three figures and an
anonymised derived CSV in the notebook session.

In [ ]:
from pathlib import Path
import base64
import hashlib
import json
import math
import urllib.request
import zlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.unicode_minus": False,
    "figure.dpi": 120,
})

## 1. Load and verify the frozen anonymised dataset

The SHA-256 check prevents a changed dataset from silently producing a different
result under the same notebook link. When run from a local repository checkout,
the notebook prefers the adjacent data file; in Colab it downloads the same file
from the public GitHub repository.

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/risk-ledger/"
    "risk-ledger-dashboard/main/"
    "chapter4_copytrading_metrics_anonymised.json"
)
EXPECTED_SHA256 = "f9f470677e1b3ec14825e14950a02e82c152f251e128dc3afab1eb1540b195ca"
LOCAL_DATA = Path("chapter4_copytrading_metrics_anonymised.json")
EMBEDDED_DATA_B64 = "eNq1XW1zHMeN/u5f0eX7clcVTjXQ73efZPk1sSJF9CW5XF25RtyROOflLjO7a5tO5b8fsLIkzjSmm82T5bLLIpfLxgINPAAeYP7xifr0Zjj2m/7Yf/rv6h+fKPXpcTxuB/rLp0+v+9vjMCmrrva3dxfHqd+MuzeKXj+NVwf1r/1uv7u7GQ/D5t8+/R3/5GHX3x6u98fv6e3O74Aa/YUOFxp/fUF/c7sdvt8Mr8fdeBz3O37Rl+N0OCrUWm2HfqP41wzTQR1O0+v+atio047+ro7Xg3r+h78q+tn+tD2q/Y/D9OM4/KSmfvcDHevtL7idxh/7qzt+2xenV9vxSl33u812OKh+GtQ03G7Pb/nTeLymvx2Gfrq6HqaL/nAY3+zoG7eH4bRhqQ7qFX2Txd3t6Xfy2/J51etpf3M+y34a34y7fnv4D0Vveny9n25Uf3W1P+2O6pvP3/6+4eer7WkzbLq3h9uON+Px/DYHPuB39C63bw95u9u+5G+oA/0mOuxIP0/nO56m3cVI4v+sbqf9z3e/o8McVX/ajEc66/D303i869TL4UAfyNvfuBkOV9N4exx/HBQJrjb784+Mm2F3HF/fqe1IP7Q5n+F36qq/pfNs1evt/qfD79Tr/Zb+hz7q7f5wGOgLt9OwGa/evnY/0ctPh/NPdp9+ov5JEn06DVf7acPC/DfJx8ZD1nNW3/f9duzPUv6L1qCeHMiOzh+C+vTdr+HvuhQ7/fbLm/FAn+PdsPl+2o/0rdQlbcPb791sNt+fPwD6uu68j4msJQWtXYSAb19DP/V92NALLgymzn34otH8VYO+i/7DV9P5q7vTdktfYmHWj4/q96fdeCsKgGbt/HxOEM9vktXgjXYQDZ3//ql+FSB1gEsBfIfp/h/rlsJ430GoSmPUZ/2h3x4lYXBNGOiA/ojKsAjowThjvbY6LmRhUfz9Y2eKARM6F5ay0OfToakKY9XTYdNLigEN68IYfGc194V5/wPvTx9sR7r68AeWhy+8IMkvqIrk1Mv9T/1OUs+6qaEFK2kHPFr+Ht2YSH9CZmp6fjydmR3Ezs0UmKmKbtsDxPLqyXYjXqHkCnIZiJJc1gPdLmMhOqONW7oAksrcOzK9aCmW9d3sNuWuwZjOzASvihjUn4c3vSChDesSkoBG1BwiuOCj51vlDLpcc8bMLtZ9EX7Vnev83GlkQkIXzUz/VSmj+vZukqQ0Jfs0BiUptcdgkrPG6mQNhOX960CvnP+thAhdcI9y6kk9nxh/CJ4jFXx69DaJBhmSNx4pHmmg/1u6wQsgdemVa2TeXaME91Vhl3Il8sK2Jhho9fSOAvfhIDrFuC6ajm7Fh0SNOtpkNJIjQZvJFjofl+KwEh6jGAD1l5FPLN0jW/QUooUB+ECHB+t08smmxeHJZtP9Dz1zFECvmHnIzMUj+S9XFQvV1/0vw1bCQHpdKohR1AlqIMfhdDRgbbCQO4eYcgABmZMj7x7rJmXUtwxCpctiCoe3DsW7ggkI97EyLCZtliq5wLBw3/ldYb95XymZYNbRr5m9oiqlVc96Sk6kwOvSOpZwgKKKjHWJcUYKib1C5r+N7zJjS7aLKEfjX23N2M7fl6sKkcCpS0oU9qPkshEKgckHMTBFkyLYFBOF4OCtybQXCbiXfLbFzsX7L8iU51Nnm0ATELo43IoGmgo+A5x3awBdm0jma+mkuccLcx1AJiIAIZSiLydvD6ZJxKA+GylZbcSF5EBEr+icI91piCEZ743LJHSz82snRN77YD+9+2KMVUmi+mJ7I0WnQmxyTsYPLgak68ma8ox1s1vWuZkbEBx8Zx8Vp5L6chRT2rieeMTOGyt6i0DgIQSCfC5EbU2G05Ojc+rCrQITfefu3ZoM6zkKFl2o+kHU6tv+NImhCsAVAIShKyP6e4pQwRIC9JZSEO2X14k+r/vGBt7loSvMXIoVwHqsxmAE8vBvdvut6AtT4RoF4+Q0BNhPESaK1tCNWsJXhxTAU1FpnS3i80D2W71QiOrZ3XSUQ1dYBbMkb3gXOMtp8EVEAg9QyhJ9JNsW4EWi1Mc0pBdo1PPt+KMchQsaotPJQMmQ+9Z0pTwlGcFkF0vP5CLjC1kWFeYxOQ/KBBFNWxKFVr0Yd5KQhUwRvfeiETpP+MIT/sWE7w31Hjy3hEVW4q15lwXaUJIwsKm3RCt06sWezi9WZqDg6Mn1iT7EWh8Ix4MO5EYoDVkISa4P7ouImQu5QLpKpliqMQvYWRXSq8v+jaRFX1CjfW9iS0hP3ttFcpGcsUCGFymZd+VcmPIQU8RV9kH+P6jL2+l0NYipcUEwh3KMjo5iHkSkAG015cmZYOyIZrUYIQeLtngFyTW4KhTGqC7vrvqb/STePL2O8bnYLEZtayLZDEbyEN4tjZI+7kpuGcn/27z0GRY/VxUsqb/0291JqukW8jNHCYqoMLJBZwx4ytPsLNj+etdM0KWiKMltyoboHF21mlRGq/8apDIABaNCwqzFMrXTBhKSmwxJ25gVmjiFL5ogWymWyzOE7vQMpVQdiAH1p1M/HX9p0xuQpxd9JOklUhBI5JnAa6EogPNqms7EpOzSr5hsepfZzD+GuhpRPd/d/Sx5kxKWTF60TeOsB0pAEQL9J2TGSYB/7kxsdunCIpZlioypS9hS+TVG/b4/yA0iXZLRibEODL61VcdVK4OZraL9WNU2Y9WX23HX6DiMl7NK5w1GDD4Zp63NG3O47DbkQJ9L+48SxKnLLTedhYSsdJNWytJWe+B0BREM+3ifpSxRl9NKQo2wlkG/v0q2CqSMV19N/W4UJSvgKAgg1t4g6eSDxeC4wBGl67OUxHU+PEolgdKt6ZWYlhS9mxZPrgMEFwBTAHLg9K/g3mZRR7j7dl50yTHgg+SK6vmrw7gZxSZdqYjhjZeLGD5atI78OkElik9Z9k/hcubUkpAhQ94P7h7gupJ6cboZr1qVFMDLWI9uTKRUGCJhCDQ5dHCCecUV0dI7hw2z4FztcVutLq97OR8uqUeGQ2TIhFwpiSTbCZiyiid2YSZAnl8ROM0Qa6J8rioHqO/2t/0vjc02cOBk95yMJ8dGXpoEwaV7Ruh8DntcZ/K03tHvaEp4Laqv+mk3iKyDkoMmWCZCHTRnxxxI1+l9NPrAOuh8VqJgo5055JxzQPZbl8Soz4bpbityQUox08rB3pvAKQRhGQK8wbRibcpqQ+bFjCVsP8OqVamsen7biyW/9d4Hl5e9jESjjxSAPPlyxjF5yrcsy6JQXvYolJd9XUFOPbl51YrFtBcRDWJ0wQdPvowJOyk3NIdF/RBcnlsd5l56CRSqEnr1dD+J2sJSi5Quk1wjo3SBYpNLaJnHE/LoY7DMEeFeQLlERt5nXumsChkIU2/kIkRBRJTTBkp8KP9jN57I9+Wwxy96784JXArti3GK0G2sh6aoXgz9JPeBS/b5nlq1CE6U7/mgHX3+xnKDIBMNO9Bl7S3ARcwSIrKpai5rk3ryRgbgBbHQrfh3um3RJfpIEiEiq5c2aXyn8wLYstAZhVa9ayutOK2ejsdJLtaCL5giQSuZ2BPJOxKmoOBlE1OY8m7ITAhnczgxh7nB5AqzswS26lIcqMtxGk8H0acUiplaNkuT9UJ0nlgARbfc1vQDjM2hejm+EVtUxpRqrybAQxoe5Coo6bnn1rP0jtKnuWfPBHH0HtX0zhn1ZHvsx8ZoxdVUsR6Zcp4BYOfLtS2uLLuy+44d2ibA56z6fNgNr1ryccrgPpRjlv3qiBECKdd6TuFiZl3GlfNxevMFTF8Kea6Pzf5A3RCdejaKbDhbQuga5ColwSVC5tZZ8n6amxEZHExC+3qumbwumb2iKpVXf9z/2LekTxdn/piYExqKvMwtI7AegjNuaaHMaCnXXnGpmczhXcRWfOGCerHf9tN4aEniL9YjMXiWMMYUDf2Tm6hZtG/Sx6rhuaie9ofjXvIhhQord5tEH0JIkFJEQ06DsIVzQjvRQRFQmNglX7pp3nX1m5VYP9vTz40RiYKnzAM0lPlqNB6RsGVOp4WlCwHBg8x7xYId2i5VXaPX6snu2E/DoaX/e7Y8BzKfLloEDsUxOofRZQXjBV7IJDPz/q/Ap7tgrVclA7LE22G77RuTryCHM2+5aWNjRI+Ez2OGlBwWHYfrbFpD9b/2fAmB6pbGtkd1eTte9SLxp6A7SrDk5r1nd2GZX0v/h2EJm1IOm86d4NwDMgshNLlAb9SLaX91J7KgrS512QKuYFunAzgK0uepkHzcIHRmlijm5EDTQRLYPQ/QjFVPpqvjaRJBrPUF+0sryiEYGBOl/wYpuFjIuoYYKLQWK3/R3dffw326d3STdvtbWZaSoSHKBHXnfSKXbiEFhxCyrIOJnXngJdyXJVEX6AlIVyXw6snV9ThtGl0BOHlmatan+9VlL/qtArLt/Aw42JiDIzfzfLaawftAkP1KjLboC3fGm5hkycgxebo3kRvudmljwOl5Mdw628VHzUD4SOD1F5GjY0uuDFcqmXQXHE+1vXXbLsOuc2ecR6HYeV2kDbCPm8O8KpT1ST05bvtDIzPCrzAZHUUicAmS0dyoyUzS2tmF+cAdyKdUuK46p0HXZAlaPetF7mIRr2otd50YDDk0XKMNBFu9NnlHTc8BD4CUV+k5dtBSZqUX4Lc+dRBAfTlMu0b3gfI0FYEuAkiBjMdFC5xG5kXO6H8TeB5QPdsfGg1Qm+Dkcp8Gyg+jDhTrSaQsJLFTzAtjXYQcqWK1KhSM+ubHu8YumpXnRClNoQQ+WOOjpiRQMrYFuJbkcOVxNrucF6tKaNXTLU+St9LRUc7fI+mEnB8l7+R5ELIedcpnPnDOfhaIAm5OWKlDoeDUd9fj4djWAT1fd3mWxZFSwUdLSNUFb4XcSeA260XwhdyddzjTZ6hmhcGrz6b+pplUsALwCG1QhhuDIycYKfbHkAWtuW5yxsqj3UJQXw/98Vo2vVSidqQIcp0lgvXm3b+YlVlM2cGB78A8SpSovtpPhyaVXKwXwSIPhQYbCS1YkyLmOQSU2+3sSuYowUXB9pbgsSpmUi+HoRXP6hXOh0VvnacoRzatAbJiEXy0icqo1cvT4bpZOyv9pmC4Scgj/zG6ZIxAqK+UG/QCc4ueITZqJ4L6dn88HVrF1MbK4YqpdyF6ujSkJJvB2WVqoSUz1POitM7JvhdMCJ8JCtWwHFF9d9qOt61A8D16W6qUci5Iju3RUxodXKbS4AThvKS2+gKKaNTn/bU8g+OLvYEgM9oAtaGTA4VbUlPWAMFFjp6MUP7yc4PM1RS6KpaIVv1t3O3G1uJXNCtYIka0gBSXwNqMQyH00y70Eh9lACl2pooeolMvhv3urv0mrdCnCG+Tl6O0EE30QRzBqRUo9QIpCL3d80KE+XWr26JX3zyiA2C0TDIgbISRh2IDo92lZwTMVeZriJZsc05+q7PfYlDfjttm2C4nTucZbTnY/uoG7byAmt8uWCT+KPALFhNWVRGjej5dXY+bRs2RdmR8G2IKLkVug/qQcgv1c3qLUGjCR9GWI3c5bm8lZdkSzSUk0WcEyjqYCqIT8ABHFAjYFoT2b4Jy+zcsV1PU5Epa/Xncb0W+H+hSdYl+1K6Uli1qzh9NoswqY8pRPJ5Vv1HkMkLltnGzvOogE/AKpXHb7CD9yoYK7mrzWJih9P494fFeAEumTP6VOteCewxz91iXk3DG9d1NI42WfLzoSQIa7mvTfzS5+5zjbP2ir5gzhQMBkZL6eH2Cbkn/k1Ev94fhpp+ao51zcgckGcDA4zd0kpiWuD4uca2Eq3S5HHDhHjDrlqx61jjXcYZYVm6EMAkLyUlH0i4GYd9SFnozwejSVy5f6Gy1Bpqc+nLY7UTqTym8oRyygd4QzvNEKTBpNe/Zl+m3ej6UPx88Te/qO6bNd3r19Jp3G7bKaOT2MCTP5FVwlLNR8uYz3vfCc2Y2SfB4BtDybW1mMX5bTWFSUJf7SV4yUM5K1yisdPGSQ2sp8BGWWcoIsUsfrXSTovqun6b+zX7Xevy4MvoL1qBNXjM/xmT7Rgzbb9E16kUvQYDHcb605AHuP6nPx22zhjAGmaSQuLoIEeikKIRvA0Jelg+1XGAXm9rcFFPUk93YXp4yciEbAcnMnEbHi6WyMlsQRg+YXmxrMD9W5QD1dP9K3tyoy/06uV6IgQJgiB4iJt6/lg9RuLWs+L1c6KqAyjUNxYJG9c1uM77ZN1K/0eDK3BghYvCEOpk6GCA9iJg6z3ryNUyUSjTZoFHPr67FuXPjS7pbAVOUYQakJIAuWwqQ8dnBrO+seAeGfWfLgwkX9Gk7X5XMqsuRInJj1QMcrGzN4qsVuYXMU1fCrhgM85aBZJShSNLi7WFYlcup/2wdI2Ecbo28KSZ4bTGQByT/EQ3kM72V+XO/2B+YXzWmv8cWcjtory6HW3kzWLES5+VSHP124HJWCsnwOEkGNBY0YydJmcnlF8TdukkG9XSYTttBHNBMRZZWWEs+Q6DwRT7E2BhMEPaNFtnHdjEwlxcYgSJgaotrkXLsaW0O1RW7yys7VlziTXXcfAl58zItV+XkdLuczr9ccBTqRpnU02m8OYjgyhTpkTLz0wZePcXXy9qks23LaVGesgIMQayyIxM02Sho9d0gjnCV6fFyiCPsyCtjCIRT2PJOKLHORwCk/RZLrgcKFYQEVblAfdlvr+TFqq5c+EnyKInnXb+UgZB6rc6JA2axG0woj6QuPQLnA+/sHKZmkE8eXqZ6JgrUhvfEWtTeY77DCGyZgMz7vYsEqQdJZdQfhsNRzr2wJBhvhBGrpxgIhgDh++CStjm1erEMUUIjdIHKE4QPEs2q54fbabhr2wKhdRAtz7O7N6RNypzJ/duslpOvlTYLDeXbsnUX5gDGVqVy6sUaR8UVgT/I+/O5qeQQYyQbjJjtKuYVL0HisMVQwfqs6KYQBgRAfhpfN2c0Sabn8ZguhS4whDO8zhTGHFcrmd58SV+UgHB1uyBAII/ezFfTcrVbk8Xx6m8m7XoHHupJi15s/xJtL3tJVaqo/jINjdvoPziv5V5VZhsawvgpMpUyPEiqefoYc8OTXlOVi9d37sR1qkWM4Y1MT2buEBpLGN+5Dyvd79fYbCVImRkB+z32NW036ry9c/qhkc4G8sIIcDGs4J6CvnKKNcyX5VRzLwT12bTf/9B4l8SL9Kgj05eqZ0T1+bA99s3lMXl3F/BQEjOfTKRo6lw+4AgulSEcgSsQqmXVp2gAGvXl/+5Fpr4pTVppeTc5JbmO9w5TCkjSZoPAtkOpR+xCuYyuFysA63ETrfpq21+NYuAMJRqkiTIgcJQbnRfbaOTZuLxRF2OxSnthQahoPgTZoFNf99MrcerAQCrXY2Uil/GOHWWy3lA6jzmPa56sC+3URc6Tb0QgnzZfBQ51W/Tqm8O2tVZLAV/u65MhOesDZT8meCs8/STOyVe5yiAIWe2FcZ2rG2AgX/xm394l0CHIbFWK3oYZaxw+6UW5P7ZSQ3ERQpJEQpuD0gfEmaieDQdxFsFCERkI7i/xgOKZ8X0O3sOFziloUu/tUfu7AJN63h/GVkb++/teDjDzptL/q9EERquX40bcMBtLVWNuQMu7hnizpw5k5ykIy998lJh+KXVOaFiErjpIDgbU5enmZmwF+F4u5TMjODFj9swissJi0jLGQtsFLCLkC49Voh8Y5l/uNlNz7F8r37xd4hKt05RhU6YpzEYU6G//Txsz6s/9djvctc7prZBJfbLgLevIn+XJ19Es+kEuxzGxvKQv55xVhbTq6V5eKl4auPZRbgEmgmjcr+UxsJiPhVIe7Hy5pMgbDcobjfkyQrXcZpz6/LRrbm2iHC61NZTAMxcWHT95Kif7IQgwIOpK6xkfECyNV0+34+vXTft+uRbg5aIUBTSL/DAmMjfAjLGCSzqUFeZXsJxOcw9tvruw2tw0QX01yZZYjD1mhYwfKR8lS/TIY1cmj5pepGPme5HIc1aPzhG/34gPZoKitcmcIp7wcDbqQJlCMNk6TyBf7StFQ95mWr5Ey5XodSmTejH14zQOH6mQo+fTkcakB2WkUK/dVIdfwWqeIr/btxelcDVORZMAeAoWPMJjk+sF37tehrLMfbiR10c+Yp2Q5i07vO7D82wQmvy5McWp1g+sy8UcixGQNdTVhHSzjsNeflhC2Y+L0iXLYzqeXLhJ2mV0Inl0YJ4JiV58XsKqN7ms4f3mh751F6P2KyOk9AlHa5Nh5k3O1nuo+VUZvmCtenHath/cy0NTLhoXmWOuDTm7LNWm/B2xOP9AUMAUQ1HWGapK6NQfh1enbSuG5adiyQx6BB8IvpJ4zmfj1bhoshopLs1lzJ8mc2EiQcOqZF79bdiNx+uWtQZM6woy1PMukKugWBSj1Q+4SjlZXlhjGmavqMIGG9ST07RvzzhwdeM0t/qdIVceCYLnm3WC0AFaFhKC4CJMtU9io/riajvethPy4spmJwLjTPuk1M7bEH2uozSfQ7Z51y6l8mrgMz0Dm6CeTepyvz0cG3drv+3yy1pj4Wyy0Rl/fhSnELQWz4gWZi/nXF5BifR5tU0cgdPqi7+fxt3+5/aRCNnTEzYHbSw5FNYuCiN9whrxVFvv5+cMh6oKHagn0yNYvijzazSv19V86XQIJmlBfyHqCt98gUq8dAfnxL6qx3RIseAxVUoZW5G+mL/H+2yctSEvLKNNtRkIsURZBVLOqL8Ou33rWv6V7AoplJxr4HSP0BhrHuD7lwNGQci0YuPlsuolpV3t6ll5YmJkhhCjQgTexfE4DP+QypJz6uthO55umkdr5L4SmDnctlljKeY5ouvydcbnL1ZP79UfprvbY7M1aVwZUw5AQAIxsgdPGh8JXjHOF/JU5eBp0OP1I9QA8iIxHpVB9OelGd5gLgfOR4GMlbp9wui4rfvjqC7H7Sizy8jrFI0KZXxnLaI+DxVanfK1QHrxQCRh8mIx+YRBsLe55dbJJC5R5j69ar/1bmWTnae35OVI/Ay9fDD0YcbHIMhXL47X6rO9zJpzvjhvIVMrCIwEZsHw2q2Yk4r9ApcJG/jTzOMKo5Fm8ZQFXw2ZvOly3O36V30rQfzDcNJyBkgnujnokcCPg3xBKc7qdh9Ym0kgo0ZsKnp5VM/6bX913fiYobc7SWVxTCJnZ+h2GU9ZoeDunHCVnKtgONu5KnvEG/Xkl9Mky2LLrXKZtuSZwOJ4GYODYGw+k2AqzxjmInmo5k6x6sy9VS/uViQrEuYCyNRoHXgFfSSfQNmgtFW7NtbJ8WAuu7AVJNZ7at6pr/rtsOubqYAruxjOzw/gEnP0JsYUcse+mKcAaSEhPubJV+C9eibviMXinJbMG8akIVF2G0NI5L5zfq1ZTPII5geLciZKg1rzJLh+z4L6cthuDrftPtDJa8Z463mwyACJ8wefpX3oK5ufbOeF6Futhvmovrq7PYgYyZZB9gp90xKA5FY1YyRpo46T+DQIUtJaBQs+qa/pr8f2CUd5i53RwRHEo3gEBOJN7u5cLCappXHVxw5xBm5lbK8eISTKiIirlI4fRw688iN/4jobW2WifcmsiRL9Lvn546WrgoK67HcbcTTElmZDCPnJexeYDo1oeXrfYcYtCMw7LWbjEOizEBhQrOaqNKi+ePWIXUJO9ur8hHlPAr37r5OWSM5UIqy5K2wNfu8xoG6QhjDT9f5NLwrnyzSQldURlOBYrwlcgnVJ6OYKIN1Jiz7mCwdI93WrszyO9EMjG5pX6cvrIjVS3KXQTJfcwQNowxdmTgt9eKwNTn3W37zaS+PCiK1sYcLevMOcknXNF9eA+634w8Grl/3xKM7uFc+t3eoWN0tJuUNyagkDPKRaZfODLzeIVeUI/DCyH5oXjcg1HsPrlSMl4s473o4oEKFcZZTIL/apirvcmhbhQIhkY9tDc/fFyNumyLSQi6SUIVE+4TNsSslCbR8aP41BYtvOy8fVXD0knha4at0NCdqK6YQJ1lqC3fSBRXQ5p7iyEfHDpJHQSEdb3w5DaZr6erjZ7q9+aFoecO5rynw2y8kfoKFECD88ParoymJnhN029gFEgMix//VruXgCMRRpUCuPZA78eHp+HDsPR0FeM53jE+PE3EEYVw6LWa8qSo2onp5uxl3ryOv7p0gsFOPI1ZHBmaA9jx7ms+aia0jlvaOh8TlXEM3bBdLtVFArj2CneC6d8MBrgChMSOG8j+7EtaNzDGTFhezLi1gV1ao/no43w5vmYbAVBQbro47AgyLePppLNEfpplq6i/xojcNB3HpQqiAbdPJmB7oHJnqnPTNoYsr2wZiQ9yoF7+3oldWze/XVuHvTvpUCZB047SgtcJ438oSYDbRAnD36+8NDQVafevKh4lMnFMbA5PD+qplPaGTak/Xcb+SbgwliFLga8x0M0q5KXCxJMtJMy3wDSZ2GF6O6HA79TXPVzssWx1Xic4048NRhzu8KSy5X7tPPVaXyBua0nKqsipnU70+74dUwTXeNTp4iX5B3Blpto7V02DMNJ6/9Q5lnDXbxWAcrCdrInUpa/ek07lqfmS5kHHPS/kefH0mgPu9XNmlQnlx6AIqRm/spAS/lIX/Nj6AwudOO5T0osIyvWWHBmQc0+hMzJjfb9sLjymPrg4nWECw/92eFvTUL9p2rbl4OIG1tdE2ZVTLq2Wm7dp2aB2GZdBsiVyaDsfDhASMfP7VNVr3c/9Tv1o4OoX103IFlEqVxiR8i+ZGYx1xQrwrj1OV238wGl59ilU8jO/hol92rL7abZmgAkFY8L6WolDAEz1tZfb6RsLI0x3U2lodD+YF1uu1GBPV1/9Pxej/tWhc/yxQfk7g16YEZ05Bnd4nSv9pWLlt5xgnybqiqYFF9tu2vflgTrf2ynw2doTGFtmhCNL/ZZU/q8/2bn/b7zcc5N13JhM4kQ27WBfdbOSnSmvqaPuxXQ3/TerflCf4HbRrI4x1/jc/6ifqfT/75yf8BsFE4jQ=="

if LOCAL_DATA.exists():
    raw = LOCAL_DATA.read_bytes()
    source_used = str(LOCAL_DATA.resolve())
else:
    # The embedded copy keeps the Drive/Colab version runnable without
    # exposing a personal repository address or relying on network access.
    raw = zlib.decompress(base64.b64decode(EMBEDDED_DATA_B64))
    source_used = "embedded frozen anonymised dataset"

observed_sha256 = hashlib.sha256(raw).hexdigest()
assert observed_sha256 == EXPECTED_SHA256, (
    "Dataset checksum mismatch. Stop and verify the archived version before analysis."
)

payload = json.loads(raw)
df = pd.DataFrame(payload["records"])

assert len(df) == 200
assert df["trader_alias"].nunique() == 200
assert df["trader_alias"].str.fullmatch(r"#\d{3} [A-Za-z]+").all()

print("Source:", source_used)
print("SHA-256:", observed_sha256)
print("Snapshot:", payload["metadata"]["snapshot_date"])
print("Captured ranking positions:", len(df))
df.head()

## 2. Reproduce the reported descriptive statistics

The maximum-decline variable is already derived from the full-precision archived
`pnlRatio` histories using the bounded proxy wealth index
$W(t)=\max[0,1+\mathrm{pnlRatio}(t)]$. The notebook checks every number stated in
the three Chapter 4 findings before drawing the figures.

In [ ]:
def sign_counts(series):
    values = series.dropna()
    return {
        "positive": int((values > 0).sum()),
        "negative": int((values < 0).sum()),
        "flat": int((values == 0).sum()),
        "n": int(len(values)),
    }


def percentile(values, proportion):
    values = sorted(float(v) for v in values)
    position = (len(values) - 1) * proportion
    lower, upper = math.floor(position), math.ceil(position)
    if lower == upper:
        return values[lower]
    return values[lower] * (upper - position) + values[upper] * (position - lower)


mdd = df["mdd_proxy"].dropna()
anchor_positive = df[df["roi_90d"] > 0]
paired = df.dropna(subset=["roi_90d", "roi_7d"]).copy()
rho = paired["roi_90d"].rank(method="average").corr(
    paired["roi_7d"].rank(method="average")
)

lower_cut = percentile(paired["roi_90d"], 1 / 3)
upper_cut = percentile(paired["roi_90d"], 2 / 3)
bottom_third = paired[paired["roi_90d"] <= lower_cut]
top_third = paired[paired["roi_90d"] >= upper_cut]

results = {
    "usable_public_curves": int(mdd.count()),
    "median_bounded_decline": float(mdd.median()),
    "decline_at_least_50pct": int((mdd >= 0.50).sum()),
    "decline_at_least_80pct": int((mdd >= 0.80).sum()),
    "roi_7d": sign_counts(df["roi_7d"]),
    "roi_30d": sign_counts(df["roi_30d"]),
    "roi_90d": sign_counts(df["roi_90d"]),
    "roi_7d_among_positive_90d": sign_counts(anchor_positive["roi_7d"]),
    "spearman_90d_vs_7d": float(rho),
    "paired_n": int(len(paired)),
    "top_third_negative_7d": [int((top_third["roi_7d"] < 0).sum()), int(len(top_third))],
    "bottom_third_negative_7d": [int((bottom_third["roi_7d"] < 0).sum()), int(len(bottom_third))],
}

assert results["usable_public_curves"] == 199
assert round(results["median_bounded_decline"] * 100, 1) == 29.8
assert results["decline_at_least_50pct"] == 51
assert results["decline_at_least_80pct"] == 15
assert results["roi_7d"] == {"positive": 115, "negative": 56, "flat": 27, "n": 198}
assert results["roi_30d"] == {"positive": 98, "negative": 78, "flat": 13, "n": 189}
assert results["roi_90d"] == {"positive": 83, "negative": 83, "flat": 5, "n": 171}
assert results["roi_7d_among_positive_90d"] == {"positive": 35, "negative": 32, "flat": 16, "n": 83}
assert round(results["spearman_90d_vs_7d"], 3) == -0.324
assert results["top_third_negative_7d"] == [27, 57]
assert results["bottom_third_negative_7d"] == [8, 58]

print(json.dumps(results, indent=2))

## 3. Figure 7 — displayed return and bounded path risk

In [ ]:
COLORS = {
    "ink": "#1F2933",
    "muted": "#687783",
    "grid": "#E2E8EC",
    "teal": "#26796A",
    "red": "#B94B45",
    "grey": "#AAB4BB",
    "light_red": "#F8E9E7",
}


def style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(color=COLORS["grid"], linewidth=0.8)
    ax.set_axisbelow(True)


usable = df.dropna(subset=["displayed_roi", "mdd_proxy"])
y = usable["mdd_proxy"] * 100
sizes = 20 + np.sqrt(usable["followers"].fillna(0)) * 8
colours = np.where(y >= 80, COLORS["red"], COLORS["teal"])

fig7, ax = plt.subplots(figsize=(10.8, 6.4))
ax.set_xscale("symlog", linthresh=0.1)
ax.axhspan(80, 100, color=COLORS["light_red"], alpha=0.65)
ax.scatter(usable["displayed_roi"], y, s=sizes, c=colours, alpha=0.72,
           edgecolor="white", linewidth=0.5)
ax.axvline(0, color=COLORS["muted"], linestyle=":", linewidth=1)
ax.axhline(80, color=COLORS["red"], linestyle="--", linewidth=1)
ax.set_ylim(-2, 103)
ax.set_xlabel("Platform-displayed cumulative ROI ratio (symlog scale)")
ax.set_ylabel("Bounded normalized decline in public pnlRatio index (%)")
ax.set_title("Displayed return and path risk are distinct", fontsize=15,
             weight="bold", pad=18)
ax.text(0.5, 1.01,
        "Platform-visibility sample; 199 usable public curves from the first 200 default-surfaced traders",
        transform=ax.transAxes, ha="center", color=COLORS["muted"], fontsize=9.5)
ax.text(0, -0.19,
        "Median decline: 29.8%. At least 50%: 51 traders; at least 80%: 15.\n"
        "The proxy is bounded at 100%; it is not audited equity and does not identify liquidation or follower loss.",
        transform=ax.transAxes, fontsize=8.4, color=COLORS["ink"], va="top")
style_axes(ax)
fig7.subplots_adjust(bottom=0.22)
plt.show()

## 4. Figure 8 — ROI signs across overlapping display windows

In [ ]:
bars = [
    ("7-day\nall visible", sign_counts(df["roi_7d"])),
    ("30-day\nall visible", sign_counts(df["roi_30d"])),
    ("90-day\nall visible", sign_counts(df["roi_90d"])),
    ("30-day\n90-day-positive", sign_counts(anchor_positive["roi_30d"])),
    ("7-day\n90-day-positive", sign_counts(anchor_positive["roi_7d"])),
]
x_positions = [0, 1, 2, 3.5, 4.5]
bottom = np.zeros(len(bars))

fig8, ax = plt.subplots(figsize=(10.8, 6.2))
for key, color in (("positive", COLORS["teal"]),
                   ("negative", COLORS["red"]),
                   ("flat", COLORS["grey"])):
    heights = np.array([count[key] for _, count in bars])
    ax.bar(x_positions, heights, width=0.72, bottom=bottom, color=color,
           edgecolor="white", label=key.title())
    for index, height in enumerate(heights):
        if height >= 8:
            ax.text(x_positions[index], bottom[index] + height / 2, str(height),
                    ha="center", va="center", color="white", weight="bold", fontsize=9)
    bottom += heights

for xpos, (_, count) in zip(x_positions, bars):
    ax.text(xpos, count["n"] + 4, f"n={count['n']}", ha="center",
            color=COLORS["muted"], fontsize=9)
ax.axvline(2.75, color=COLORS["grid"], linewidth=1.5)
ax.set_xticks(x_positions, [label for label, _ in bars])
ax.set_ylabel("Number of traders")
ax.set_title("The sign of observed ROI varies across overlapping display windows",
             fontsize=14.5, weight="bold", pad=18)
ax.text(0.5, 1.01,
        "Descriptive counts only; the chart is not a test against a 50% market baseline",
        transform=ax.transAxes, ha="center", color=COLORS["muted"], fontsize=9.5)
ax.legend(frameon=False, ncol=3, loc="upper right")
style_axes(ax)
plt.show()

## 5. Figure 9 — 90-day versus rolling 7-day ROI

In [ ]:
colours = np.where(paired["roi_7d"] < 0, COLORS["red"], COLORS["grey"])
sizes = 22 + np.sqrt(paired["followers"].fillna(0)) * 9

fig9, ax = plt.subplots(figsize=(10.8, 6.4))
ax.set_xscale("symlog", linthresh=10)
ax.set_yscale("symlog", linthresh=10)
ax.scatter(paired["roi_90d"], paired["roi_7d"], s=sizes, c=colours,
           alpha=0.72, edgecolor="white", linewidth=0.5)
ax.axhline(0, color=COLORS["ink"], linestyle="--", linewidth=1)
ax.axvline(0, color=COLORS["muted"], linestyle=":", linewidth=1)
ax.set_xlabel("Observed 90-day ROI (%)")
ax.set_ylabel("Observed rolling 7-day ROI (%)")
ax.set_title("An inverse cross-sectional association between overlapping ROI windows",
             fontsize=14.2, weight="bold", pad=18)
ax.text(0.5, 1.01,
        f"Spearman rho = {rho:.3f}, n = {len(paired)}; this is not an out-of-sample prediction test",
        transform=ax.transAxes, ha="center", color=COLORS["muted"], fontsize=9.5)
ax.text(0, -0.19,
        "Top third by 90-day ROI: 47.4% had a negative 7-day ROI (27/57). "
        "Bottom third: 13.8% (8/58).\n"
        "The windows overlap and the data concern lead-trader ROI, not realised follower outcomes.",
        transform=ax.transAxes, fontsize=8.5, color=COLORS["ink"], va="top")
style_axes(ax)
fig9.subplots_adjust(bottom=0.22)
plt.show()

## 6. Save reproducibility outputs

These files are generated inside the Colab runtime. They contain only researcher-assigned
pseudonyms and aggregate/derived measures. Colab runtimes are temporary, so download
anything you want to retain before closing the session.

In [ ]:
output_dir = Path("chapter4_outputs")
output_dir.mkdir(exist_ok=True)

fig7.savefig(output_dir / "figure_7_displayed_roi_vs_bounded_drawdown.png",
             dpi=240, bbox_inches="tight", facecolor="white")
fig8.savefig(output_dir / "figure_8_roi_signs_by_window.png",
             dpi=240, bbox_inches="tight", facecolor="white")
fig9.savefig(output_dir / "figure_9_90d_vs_7d_association.png",
             dpi=240, bbox_inches="tight", facecolor="white")
df.to_csv(output_dir / "chapter4_copytrading_metrics_anonymised.csv", index=False)

for path in sorted(output_dir.iterdir()):
    print(path, f"({path.stat().st_size:,} bytes)")